# 🧪 本地驗證：不靠提交，就知道改進版有沒有變好

**這份 notebook 的用途**：用**訓練集的標準答案（`.geff`）**，在本地（其實是在 Kaggle 上、但用 train 標籤而非提交）幫 baseline 與改進版各打一個分數，直接比較。這樣每改一版**不用浪費提交次數**就知道方向對不對——這是把分數穩定往上推的關鍵習慣（立霧第 8 點）。

> ⚠️ **這是「代理分數（proxy）」、不是排行榜真分數**。官方頁面沒完整公開兩個細節：①node 數量過度膨脹的懲罰公式；②稀疏標註的精確處理。所以這裡算的是**忠實近似**——它能可靠告訴你「**變好還是變壞**」的方向，但數字不會跟排行榜一模一樣。

> 🌐 **跑這份要把 Internet 開著**（右側 Settings → Internet On）。因為要 `pip install` 新版 zarr 來讀 v3 格式的答案檔。**這份不是拿去提交的**，所以開網路沒問題。

**代理分數怎麼算**（對齊官方描述）：
- 每個時間點，把預測的細胞用匈牙利演算法配到最近的真值細胞（距離 ≤ 7 µm 才算配上）。
- 一條預測連線，若兩端都配到真值、且真值圖也有這條邊 → **TP**；配到但真值沒這條 → **FP**；真值有但沒被預測到 → **FN**。
- `edge_jaccard = TP / (TP+FP+FN)`；`division_jaccard` 同理算「分裂節點」；`proxy = edge_j + 0.1 × div_j`。

In [ ]:
# 確保有 zarr>=3（讀 v3 格式的影像與答案檔）。需 Internet On。
import importlib, subprocess, sys
def ensure_zarr():
    try:
        import zarr
        if int(zarr.__version__.split('.')[0]) >= 3:
            return zarr
    except Exception:
        pass
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'zarr>=3'])
    importlib.invalidate_caches()
    import zarr
    return zarr
zarr = ensure_zarr()
print('zarr', zarr.__version__)

In [ ]:
import os
from collections import defaultdict
import numpy as np
from scipy.ndimage import uniform_filter, label, distance_transform_edt
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

TRAIN = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
SCALE = np.array([1.625, 0.40625, 0.40625])
DOWNSAMPLE = 4
PERCENTILE = 90
MAX_LINK_DISTANCE = 15.0
DIV_DISTANCE = 8.0
GAP_DISTANCE = 20.0
WS_MIN_DISTANCE = 2
MATCH_DISTANCE = 7.0          # µm：評分時預測↔真值的配對上限（官方規定）
MAX_SAMPLES = 2              # 先驗 2 個訓練樣本（要更穩可加大，但會更慢）

def scaled_pairwise(A, B):
    diff = A[:, None, :] - B[None, :, :]
    return np.sqrt(((diff * SCALE) ** 2).sum(axis=2))

print('設定完成；TRAIN =', TRAIN)

In [ ]:
# === 讀真值（.geff）與影像（.zarr）===
def read_geff(geff_path):
    g = zarr.open(geff_path, mode='r')
    ids = np.asarray(g['nodes/ids'][:])
    t = np.asarray(g['nodes/props/t/values'][:])
    z = np.asarray(g['nodes/props/z/values'][:])
    y = np.asarray(g['nodes/props/y/values'][:])
    x = np.asarray(g['nodes/props/x/values'][:])
    edges = np.asarray(g['edges/ids'][:])               # (N, 2) = (source, target)
    nodes = {int(i): (int(tt), float(zz), float(yy), float(xx))
             for i, tt, zz, yy, xx in zip(ids, t, z, y, x)}
    edge_list = [(int(a), int(b)) for a, b in edges]
    return nodes, edge_list


def open_image(zarr_path):
    return zarr.open(zarr_path, mode='r')['0']           # arr[t] -> (Z,Y,X); arr.shape=(T,Z,Y,X)


# 先看一個 .geff 的結構，確認路徑沒猜錯
_names = sorted(d[:-5] for d in os.listdir(TRAIN) if d.endswith('.geff'))
print('訓練樣本數：', len(_names))
_gn, _ge = read_geff(os.path.join(TRAIN, _names[0] + '.geff'))
print(f'樣本 {_names[0]}：真值 {len(_gn)} 個節點、{len(_ge)} 條邊')

In [ ]:
# === 兩種偵測 ===
def detect_baseline(vol):
    ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
    sm = uniform_filter(ds.astype(np.float32), size=3)
    binary = sm > np.percentile(sm, PERCENTILE)
    lab, n = label(binary)
    out = []
    for i in range(1, n + 1):
        c = np.argwhere(lab == i)
        if len(c):
            out.append(c.mean(axis=0) * DOWNSAMPLE)
    return out

def detect_watershed(vol):
    ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
    sm = uniform_filter(ds.astype(np.float32), size=3)
    binary = sm > np.percentile(sm, PERCENTILE)
    if not binary.any():
        return []
    dist = distance_transform_edt(binary)
    peaks = peak_local_max(dist, min_distance=WS_MIN_DISTANCE, labels=binary)
    if len(peaks) == 0:
        return []
    markers = np.zeros(dist.shape, dtype=np.int32)
    markers[tuple(peaks.T)] = np.arange(1, len(peaks) + 1)
    lab = watershed(-dist, markers, mask=binary)
    out = []
    for i in range(1, int(lab.max()) + 1):
        c = np.argwhere(lab == i)
        if len(c):
            out.append(c.mean(axis=0) * DOWNSAMPLE)
    return out


# === 一條完整 pipeline（偵測 + 連線；improved=True 才加分裂與補軌）===
def run_pipeline(arr, n_t, detect_fn, improved):
    nodes, edges = {}, []
    frame_ids, frame_xyz = [], []
    nid = 1
    for t in range(n_t):
        cents = detect_fn(np.asarray(arr[t]))
        ids, xyz = [], []
        for c in cents:
            nodes[nid] = (t, float(c[0]), float(c[1]), float(c[2]))
            ids.append(nid); xyz.append(c); nid += 1
        frame_ids.append(ids)
        frame_xyz.append(np.array(xyz) if xyz else np.empty((0, 3)))

    has_in, out_count = set(), defaultdict(int)
    for t in range(n_t - 1):
        pid, pc = frame_ids[t], frame_xyz[t]
        cid, cc = frame_ids[t + 1], frame_xyz[t + 1]
        if len(pid) == 0 or len(cid) == 0:
            continue
        D = scaled_pairwise(pc, cc)
        rows, cols = linear_sum_assignment(D)
        mp, mc = set(), set()
        for ri, ci in zip(rows, cols):
            if D[ri, ci] <= MAX_LINK_DISTANCE:
                edges.append((pid[ri], cid[ci])); mp.add(ri); mc.add(ci)
                has_in.add(cid[ci]); out_count[pid[ri]] += 1
        if improved:
            for ci in range(len(cid)):
                if ci in mc:
                    continue
                d = np.sqrt((((pc - cc[ci]) * SCALE) ** 2).sum(axis=1))
                j = int(np.argmin(d))
                if j in mp and d[j] <= DIV_DISTANCE and out_count[pid[j]] < 2:
                    edges.append((pid[j], cid[ci])); has_in.add(cid[ci]); out_count[pid[j]] += 1
    if improved:
        has_out = set(out_count.keys())
        for t in range(n_t - 2):
            ends = [(i, n_) for i, n_ in enumerate(frame_ids[t]) if n_ not in has_out]
            starts = [(j, n_) for j, n_ in enumerate(frame_ids[t + 2]) if n_ not in has_in]
            if not ends or not starts:
                continue
            ec = frame_xyz[t][[i for i, _ in ends]]
            sc = frame_xyz[t + 2][[j for j, _ in starts]]
            D = scaled_pairwise(ec, sc)
            rows, cols = linear_sum_assignment(D)
            for ri, ci in zip(rows, cols):
                if D[ri, ci] <= GAP_DISTANCE:
                    edges.append((ends[ri][1], starts[ci][1]))
                    has_out.add(ends[ri][1]); has_in.add(starts[ci][1])
    return nodes, edges

print('pipeline 定義完成')

In [ ]:
# === 代理評分器（對齊官方描述的近似）===
def _outdeg(edges):
    d = defaultdict(int)
    for s, _ in edges:
        d[s] += 1
    return d

def score(pred_nodes, pred_edges, gt_nodes, gt_edges):
    pred_by_t, gt_by_t = defaultdict(list), defaultdict(list)
    for nid, (t, z, y, x) in pred_nodes.items():
        pred_by_t[t].append(nid)
    for gid, (t, z, y, x) in gt_nodes.items():
        gt_by_t[t].append(gid)

    match = {}   # pred_node_id -> gt_node_id（每幀匈牙利配對，≤7µm）
    for t in gt_by_t:
        P, G = pred_by_t.get(t, []), gt_by_t[t]
        if not P or not G:
            continue
        Pc = np.array([pred_nodes[p][1:] for p in P])
        Gc = np.array([gt_nodes[g][1:] for g in G])
        D = scaled_pairwise(Pc, Gc)
        rows, cols = linear_sum_assignment(D)
        for ri, ci in zip(rows, cols):
            if D[ri, ci] <= MATCH_DISTANCE:
                match[P[ri]] = G[ci]

    gt_edge_set = set(gt_edges)
    pred_mapped = set()
    for s, t in pred_edges:
        if s in match and t in match:       # 稀疏感知：只看兩端都配到真值的邊
            pred_mapped.add((match[s], match[t]))
    TP = len(pred_mapped & gt_edge_set)
    FP = len(pred_mapped - gt_edge_set)
    FN = len(gt_edge_set - pred_mapped)
    edge_j = TP / (TP + FP + FN) if (TP + FP + FN) else 0.0

    god, pod = _outdeg(gt_edges), _outdeg(pred_edges)
    gt_div = {g for g in god if god[g] >= 2}
    pred_div = {match[p] for p in pod if pod[p] >= 2 and p in match}
    dTP = len(pred_div & gt_div); dFP = len(pred_div - gt_div); dFN = len(gt_div - pred_div)
    div_j = dTP / (dTP + dFP + dFN) if (dTP + dFP + dFN) else 0.0

    return dict(edge_j=edge_j, div_j=div_j, proxy=edge_j + 0.1 * div_j,
                n_pred=len(pred_nodes), n_gt=len(gt_nodes), matched=len(match),
                gt_div=len(gt_div), pred_div=len(pred_div))

print('評分器定義完成')

In [ ]:
# === 跑對照：baseline vs 改進版，各打分 ===
samples = sorted(d[:-5] for d in os.listdir(TRAIN) if d.endswith('.geff'))[:MAX_SAMPLES]
print('驗證樣本：', samples, '\n')

rows = []
for name in samples:
    arr = open_image(os.path.join(TRAIN, name + '.zarr'))
    n_t = arr.shape[0]
    gt_nodes, gt_edges = read_geff(os.path.join(TRAIN, name + '.geff'))
    for tag, detect, improved in [('baseline', detect_baseline, False),
                                  ('improved', detect_watershed, True)]:
        pn, pe = run_pipeline(arr, n_t, detect, improved)
        s = score(pn, pe, gt_nodes, gt_edges)
        rows.append((name, tag, s))
        print(f'{name} [{tag:8}] proxy={s["proxy"]:.3f}  edge_j={s["edge_j"]:.3f}  div_j={s["div_j"]:.3f}'
              f'  | pred_nodes={s["n_pred"]} gt={s["n_gt"]} matched={s["matched"]}'
              f'  pred_div={s["pred_div"]} gt_div={s["gt_div"]}')

print('\n=== 平均 proxy 分數 ===')
for tag in ['baseline', 'improved']:
    vals = [s['proxy'] for nm, tg, s in rows if tg == tag]
    print(f'{tag:8}: {np.mean(vals):.3f}')

## 怎麼看這個結果

- **`proxy` 改進版 > baseline** → 方向對了，這版有變好，可以考慮提交。
- **`proxy` 改進版 < baseline** → 過頭了（很可能是 watershed 過度切割、`pred_nodes` 爆增）。這時調 `PERCENTILE`（拉高，如 95）或 `WS_MIN_DISTANCE`（加大），再跑一次這份驗證——**完全不用浪費提交次數**。
- 看 `matched`（配到真值的數）和 `pred_div / gt_div`：分裂偵測有沒有對到真的分裂。

> 再次提醒：這是**代理分數**，跟排行榜不會一模一樣（少了官方的 node 膨脹懲罰細節）。但它**很適合拿來「改一版、比一版」**，這就是把分數穩定往上推、又不燒提交次數的正確做法。